In [1]:
import re
import pandas as pd
import warnings
from sqlalchemy import create_engine, inspect
from concurrent.futures import ThreadPoolExecutor
import os
from sqlalchemy import inspect
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

In [2]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [3]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Datos de conexión
server = "Servidor"
database = "Databese Usado"
username = "usuario asignado"
password = "la contraseña :)"

# Codificar el driver correctamente
driver = quote_plus("ODBC Driver 17 for SQL Server")

# Crear cadena de conexión
connection_string = f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver={driver}"

# Crear engine
engine = create_engine(connection_string)

# Probar conexión
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Conexión exitosa:", result.fetchone())
except Exception as e:
    print("Error de conexión:", e)


Conexión exitosa: (1,)


In [4]:
query = """
SELECT *
FROM [CONTROL_PYMES].[dbo].[ocu_202603]
WHERE FIRST_LEVEL IN ('FIJO', 'MOVIL');
"""

movil = pd.read_sql(query, engine)

# Revisar resultados
print(movil.head(10))
print(movil.columns)

                        PRODUCTO                        SUBPRODUCTO  \
0                       INTERNET                             500 Mb   
1             TELEVISIÓN DIGITAL      TV AVANZADO 2 DECOS FULL TIGO   
2                       INTERNET                             500 Mb   
3                       INTERNET                             500 Mb   
4  TELEFONÍA LOCAL - TOIP - VOIP  PLAN ILIMITADO SIN FRONTERAS PLUS   
5  TELEFONÍA LOCAL - TOIP - VOIP  PLAN ILIMITADO SIN FRONTERAS PLUS   
6                       INTERNET                             120 Mb   
7          CONECTIVIDAD NACIONAL                               2 Mb   
8                    TRONCAL SIP                            0.117MB   
9           TELEVISIÓN ANALÓGICA                         Television   

                USO                                            CLIENTE  \
0       RESIDENCIAL                       GARCES SANTA LEIDY ALEJANDRA   
1       RESIDENCIAL                       GARCES SANTA LEIDY ALEJANDRA

In [6]:
movil.to_csv("OCU_MARZO.csv")

In [5]:
query = """
SELECT *
FROM [CONTROL_PYMES].[dbo].[FACT_202603]
WHERE FIRST_LEVEL IN ('FIJO', 'MOVIL');
"""

facturacion = pd.read_sql(query, engine)

# Revisar resultados
print(facturacion.head(10))
print(facturacion.columns)

           NIT                                             Nombre Suscripcion  \
0   8110416157                                    HWARANGDO EAT      20797737   
1   8600123401  FRACO FABRICA COLOMBIANA DE REPUESTOS AUTOMOTO...    16954820   
2   8600123401  FRACO FABRICA COLOMBIANA DE REPUESTOS AUTOMOTO...    16954820   
3   8600123401  FRACO FABRICA COLOMBIANA DE REPUESTOS AUTOMOTO...    16954820   
4   8600123401  FRACO FABRICA COLOMBIANA DE REPUESTOS AUTOMOTO...    16954820   
5   8110274881                               CONSTRUBIENES SAS       14132573   
6  10141872968               CALIXTO TARAZONA JHONATTAN REMBRANDT    12322624   
7  10141872968               CALIXTO TARAZONA JHONATTAN REMBRANDT    12322624   
8  10375740689                          ALVAREZ VANEGAS JULIANA      13304411   
9  10375740689                          ALVAREZ VANEGAS JULIANA      13304411   

  strServicioSuscrito Codigo_Servicio Descripcion_Servicio  \
0           141148276            3307     RECA

In [ ]:
facturacion.to_csv("FACT_MARZO.csv")

# FACT_2026 :)

In [ ]:

# ==========================================
# 🔹 Función para leer una tabla por chunks
# ==========================================
def read_table_in_chunks(table, engine, chunksize=500_000):
    """
    Lee una tabla SQL Server por partes (chunks) para evitar sobrecargar la memoria.
    Agrega metadatos de fuente y periodo.
    """
    dfs = []

    with engine.connect() as conn:
        query = f"SELECT * FROM [dbo].[{table}]"
        print(f"\n➡️ Leyendo tabla: {table} ...")

        # Detectar periodo a partir del nombre de la tabla
        match = re.search(r"(20\d{2})[_\-]?(\d{2})", table)
        if match:
            year, month = match.groups()
            periodo_detectado = pd.to_datetime(
                f"{year}-{month}-01",
                format="%Y-%m-%d",
                errors="coerce"
            )
            print(f"   📅 Periodo detectado: {periodo_detectado.strftime('%Y-%m-%d')}")
        else:
            periodo_detectado = pd.NaT
            print("   ⚠️ No se detectó periodo en el nombre de la tabla.")

        # Lectura por chunks
        for i, chunk in enumerate(pd.read_sql(query, conn, chunksize=chunksize)):
            print(f"   🧩 Chunk {i+1} leído ({len(chunk):,} filas)")
            chunk["SOURCE_TABLE"] = table
            chunk["PERIODO"] = periodo_detectado
            dfs.append(chunk)

    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"✅ Tabla {table} completada: {len(df_final):,} filas.")
        return df_final
    else:
        print(f"⚠️ Tabla {table} sin datos.")
        return pd.DataFrame()


# ==========================================
# 🔹 Función para leer tablas por prefijo en paralelo
# ==========================================
def read_tables_parallel(prefix, engine, max_workers=4, chunksize=500_000):
    """
    Busca todas las tablas con un prefijo dado (ej: FACT_2026)
    y las carga en paralelo.
    """
    inspector = inspect(engine)
    all_tables = inspector.get_table_names(schema="dbo")

    matching_tables = [
        t for t in all_tables
        if t.startswith(prefix) and not t.lower().endswith("bk")
    ]

    if not matching_tables:
        print(f"⚠️ No se encontraron tablas con el prefijo '{prefix}'.")
        return pd.DataFrame()

    print(f"\n📦 Se encontraron {len(matching_tables)} tablas para '{prefix}':")
    for t in matching_tables:
        print(f"   • {t}")
    print("==============================================")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(
            executor.map(
                lambda t: read_table_in_chunks(t, engine, chunksize),
                matching_tables
            )
        )

    df_total = pd.concat(dfs, ignore_index=True)
    print(f"\n🚀 Carga completada para '{prefix}': {len(df_total):,} filas.\n")
    return df_total


# ==========================================
# 🔹 Carga FACT 2026 y exporta a PARQUET
# ==========================================
def load_fact_2026(engine, max_workers=4, chunksize=250_000, output_dir="exports"):
    """
    Carga las tablas FACT_2026 desde SQL Server
    y las exporta a formato PARQUET.
    """
    os.makedirs(output_dir, exist_ok=True)

    print("\n================== CARGA DE FACT 2026 ==================")

    fact_2026 = read_tables_parallel(
        prefix="FACT_2026",
        engine=engine,
        max_workers=max_workers,
        chunksize=chunksize
    )

    if fact_2026.empty:
        print("⚠️ FACT 2026 no tiene datos. No se genera archivo.")
        return fact_2026

    fact_path = os.path.join(output_dir, "FACT_2026_TOTAL.parquet")

    fact_2026.to_parquet(
        fact_path,
        engine="pyarrow",
        index=False
    )

    print(f"💾 FACT 2026 exportado a: {fact_path}")
    print(f"📊 Total filas FACT 2026: {len(fact_2026):,}")

    # Resumen
    resumen_path = os.path.join(output_dir, "Resumen_FACT_2026.txt")
    with open(resumen_path, "w", encoding="utf-8") as f:
        f.write("===== RESUMEN CARGA FACT 2026 =====\n")
        f.write(f"Total filas cargadas: {len(fact_2026):,}\n")
        f.write(f"Archivo exportado: {fact_path}\n")

    print(f"📝 Resumen guardado en: {resumen_path}")

    return fact_2026


# ==========================================
# 🔹 EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    fact_2026 = load_fact_2026(
        engine=engine,
        max_workers=4,
        chunksize=250_000,
        output_dir="exports"
    )


In [ ]:
# ==========================================
# 🔹 Función para leer una tabla por chunks
# ==========================================
def read_table_in_chunks(table, engine, chunksize=500_000):
    """
    Lee una tabla SQL Server por partes (chunks) para evitar sobrecargar la memoria.
    Agrega metadatos de fuente y periodo.
    """
    dfs = []

    with engine.connect() as conn:
        query = f"SELECT * FROM [dbo].[{table}]"
        print(f"\n➡️ Leyendo tabla: {table} ...")

        # Detectar periodo a partir del nombre de la tabla
        match = re.search(r"(20\d{2})[_\-]?(\d{2})", table)
        if match:
            year, month = match.groups()
            periodo_detectado = pd.to_datetime(
                f"{year}-{month}-01",
                format="%Y-%m-%d",
                errors="coerce"
            )
            print(f"   📅 Periodo detectado: {periodo_detectado.strftime('%Y-%m-%d')}")
        else:
            periodo_detectado = pd.NaT
            print("   ⚠️ No se detectó periodo en el nombre de la tabla.")

        # Lectura por chunks
        for i, chunk in enumerate(pd.read_sql(query, conn, chunksize=chunksize)):
            print(f"   🧩 Chunk {i+1} leído ({len(chunk):,} filas)")
            chunk["SOURCE_TABLE"] = table
            chunk["PERIODO"] = periodo_detectado
            dfs.append(chunk)

    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"✅ Tabla {table} completada: {len(df_final):,} filas.")
        return df_final
    else:
        print(f"⚠️ Tabla {table} sin datos.")
        return pd.DataFrame()


# ==========================================
# 🔹 Función para leer tablas por prefijo en paralelo
# ==========================================
def read_tables_parallel(prefix, engine, max_workers=4, chunksize=500_000):
    """
    Busca todas las tablas con un prefijo dado (ej: FACT_2025)
    y las carga en paralelo.
    """
    inspector = inspect(engine)
    all_tables = inspector.get_table_names(schema="dbo")

    matching_tables = [
        t for t in all_tables
        if t.startswith(prefix) and not t.lower().endswith("bk")
    ]

    if not matching_tables:
        print(f"⚠️ No se encontraron tablas con el prefijo '{prefix}'.")
        return pd.DataFrame()

    print(f"\n📦 Se encontraron {len(matching_tables)} tablas para '{prefix}':")
    for t in matching_tables:
        print(f"   • {t}")
    print("==============================================")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(
            executor.map(
                lambda t: read_table_in_chunks(t, engine, chunksize),
                matching_tables
            )
        )

    df_total = pd.concat(dfs, ignore_index=True)
    print(f"\n🚀 Carga completada para '{prefix}': {len(df_total):,} filas.\n")
    return df_total


# ==========================================
# 🔹 Carga FACT 2025 y exporta a PARQUET
# ==========================================
def load_fact_2025(engine, max_workers=4, chunksize=350_000, output_dir="exports"):
    """
    Carga las tablas FACT_2025 desde SQL Server
    y las exporta a formato PARQUET.
    """
    os.makedirs(output_dir, exist_ok=True)

    print("\n================== CARGA DE FACT 2025 ==================")

    fact_2025 = read_tables_parallel(
        prefix="FACT_2025",
        engine=engine,
        max_workers=max_workers,
        chunksize=chunksize
    )

    if fact_2025.empty:
        print("⚠️ FACT 2025 no tiene datos. No se genera archivo.")
        return fact_2025

    fact_path = os.path.join(output_dir, "FACT_2025_TOTAL.parquet")

    fact_2025.to_parquet(
        fact_path,
        engine="pyarrow",
        index=False
    )

    print(f"💾 FACT 2025 exportado a: {fact_path}")
    print(f"📊 Total filas FACT 2025: {len(fact_2025):,}")

    # Resumen
    resumen_path = os.path.join(output_dir, "Resumen_FACT_2025.txt")
    with open(resumen_path, "w", encoding="utf-8") as f:
        f.write("===== RESUMEN CARGA FACT 2025 =====\n")
        f.write(f"Total filas cargadas: {len(fact_2025):,}\n")
        f.write(f"Archivo exportado: {fact_path}\n")

    print(f"📝 Resumen guardado en: {resumen_path}")

    return fact_2025


# ==========================================
# 🔹 EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    fact_2025 = load_fact_2025(
        engine=engine,
        max_workers=4,
        chunksize=350_000,
        output_dir="exports"
    )

In [ ]:
# ==========================================
# 🔹 Función para leer una tabla por chunks
# ==========================================
def read_table_in_chunks(table, engine, chunksize=500_000):
    """
    Lee una tabla SQL Server por partes (chunks) para evitar sobrecargar la memoria.
    Agrega metadatos de fuente y periodo.
    """
    dfs = []

    with engine.connect() as conn:
        query = f"SELECT * FROM [dbo].[{table}]"
        print(f"\n➡️ Leyendo tabla: {table} ...")

        # Detectar periodo desde el nombre de la tabla
        match = re.search(r"(20\d{2})[_\-]?(\d{2})", table)
        if match:
            year, month = match.groups()
            periodo_detectado = pd.to_datetime(
                f"{year}-{month}-01",
                format="%Y-%m-%d",
                errors="coerce"
            )
            print(f"   📅 Periodo detectado: {periodo_detectado.strftime('%Y-%m-%d')}")
        else:
            periodo_detectado = pd.NaT
            print("   ⚠️ No se detectó periodo en el nombre de la tabla.")

        # Lectura por chunks
        for i, chunk in enumerate(pd.read_sql(query, conn, chunksize=chunksize)):
            print(f"   🧩 Chunk {i+1} leído ({len(chunk):,} filas)")
            chunk["SOURCE_TABLE"] = table
            chunk["PERIODO"] = periodo_detectado
            dfs.append(chunk)

    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"✅ Tabla {table} completada: {len(df_final):,} filas totales.")
        return df_final
    else:
        print(f"⚠️ Tabla {table} sin datos.")
        return pd.DataFrame()


# ==========================================
# 🔹 Función para leer tablas por prefijo en paralelo
# ==========================================
def read_tables_parallel(prefix, engine, max_workers=4, chunksize=500_000):
    """
    Busca todas las tablas con un prefijo dado (ej: OCU_2025)
    y las carga en paralelo.
    """
    inspector = inspect(engine)
    all_tables = inspector.get_table_names(schema="dbo")

    matching_tables = [
        t for t in all_tables
        if t.startswith(prefix) and not t.lower().endswith("bk")
    ]

    if not matching_tables:
        print(f"⚠️ No se encontraron tablas con el prefijo '{prefix}'.")
        return pd.DataFrame()

    print(f"\n📦 Se encontraron {len(matching_tables)} tablas para '{prefix}':")
    for t in matching_tables:
        print(f"   • {t}")
    print("==============================================")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(
            executor.map(
                lambda t: read_table_in_chunks(t, engine, chunksize),
                matching_tables
            )
        )

    df_total = pd.concat(dfs, ignore_index=True)
    print(f"\n🚀 Carga completada para '{prefix}': {len(df_total):,} filas totales.\n")
    return df_total


# ==========================================
# 🔹 Cargar OCU_2025 y exportar a PARQUET
# ==========================================
def load_ocu_2025_parquet(engine, max_workers=4, chunksize=250_000, output_dir="exports"):
    """
    Carga las tablas OCU_2025 desde SQL Server
    y las exporta a formato PARQUET.
    """
    os.makedirs(output_dir, exist_ok=True)

    print("\n================== CARGA DE OCU 2025 ==================")

    ocu_2025_total = read_tables_parallel(
        prefix="OCU_2025",
        engine=engine,
        max_workers=max_workers,
        chunksize=chunksize
    )

    if ocu_2025_total.empty:
        print("⚠️ OCU_2025 no tiene datos. No se genera archivo.")
        return ocu_2025_total

    ocu_path = os.path.join(output_dir, "OCU_2025_TOTAL.parquet")

    ocu_2025_total.to_parquet(
        ocu_path,
        engine="pyarrow",
        index=False
    )

    print(f"💾 OCU_2025 exportado a: {ocu_path}")
    print(f"📊 Total filas OCU_2025: {len(ocu_2025_total):,}")

    # Resumen
    resumen_path = os.path.join(output_dir, "Resumen_OCU_2025.txt")
    with open(resumen_path, "w", encoding="utf-8") as f:
        f.write("===== RESUMEN CARGA OCU_2025 =====\n")
        f.write(f"Total filas cargadas: {len(ocu_2025_total):,}\n")
        f.write(f"Archivo exportado: {ocu_path}\n")

    print(f"📝 Resumen guardado en: {resumen_path}")

    return ocu_2025_total


# ==========================================
# 🔹 EJECUCIÓN SOLO OCU_2025
# ==========================================
if __name__ == "__main__":
    ocu_2025_total = load_ocu_2025_parquet(
        engine=engine,
        max_workers=4,
        chunksize=250_000,
        output_dir="exports"
    )

## ocu 2026

In [ ]:
import pandas as pd
import os
import re
from sqlalchemy import inspect
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# 🔹 NORMALIZACIÓN DE COLUMNAS
# ==========================================
def normalize_columns(chunk):

    upper_map = {}
    seen = set()

    for col in chunk.columns:
        col_upper = col.upper()

        # evitar duplicados tipo first_level vs FIRST_LEVEL
        if col_upper in seen:
            continue

        upper_map[col] = col_upper
        seen.add(col_upper)

    chunk.rename(columns=upper_map, inplace=True)

    return chunk


# ==========================================
# 🔹 FILTRO ESTRICTO (SOLO 2026XX)
# ==========================================
def filter_tables(all_tables, year="2026"):

    pattern = re.compile(rf"^ocu_{year}\d{{2}}$", re.IGNORECASE)

    valid_tables = [t for t in all_tables if pattern.match(t)]

    return sorted(valid_tables)


# ==========================================
# 🔹 LECTURA POR CHUNKS
# ==========================================
def read_table_in_chunks(table, engine, chunksize=500_000):

    dfs = []

    with engine.connect() as conn:

        query = f"SELECT * FROM [dbo].[{table}]"
        print(f"\n➡️ Leyendo tabla: {table} ...")

        # detectar periodo
        match = re.search(r"(20\d{2})(\d{2})", table)
        if match:
            year, month = match.groups()
            periodo_detectado = pd.to_datetime(f"{year}-{month}-01")
            print(f"   📅 Periodo detectado: {periodo_detectado}")
        else:
            periodo_detectado = pd.NaT
            print("   ⚠️ No se detectó periodo")

        for i, chunk in enumerate(pd.read_sql(query, conn, chunksize=chunksize)):

            print(f"   🧩 Chunk {i+1} ({len(chunk):,} filas)")

            chunk = normalize_columns(chunk)

            chunk["SOURCE_TABLE"] = table
            chunk["PERIODO"] = periodo_detectado

            dfs.append(chunk)

    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"✅ Tabla {table}: {len(df_final):,} filas")
        return df_final

    return pd.DataFrame()


# ==========================================
# 🔹 LECTURA PARALELA
# ==========================================
def read_tables_parallel(engine, year="2026", max_workers=4, chunksize=500_000):

    inspector = inspect(engine)
    all_tables = inspector.get_table_names(schema="dbo")

    matching_tables = filter_tables(all_tables, year)

    if not matching_tables:
        print(f"⚠️ No hay tablas válidas para el año {year}")
        return pd.DataFrame()

    print(f"\n📦 Tablas válidas ({len(matching_tables)}):")
    for t in matching_tables:
        print(f"   • {t}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(
            executor.map(
                lambda t: read_table_in_chunks(t, engine, chunksize),
                matching_tables
            )
        )

    df_total = pd.concat(dfs, ignore_index=True)

    print(f"\n🚀 Total cargado: {len(df_total):,} filas")

    return df_total


# ==========================================
# 🔹 VALIDACIÓN
# ==========================================
def validate_data(df):

    print("\n📊 VALIDACIÓN DE DATOS")

    resumen = (
        df.groupby("PERIODO", dropna=False)
        .agg(
            registros=("CLIENTE_ID", "count"),
            first_level=("FIRST_LEVEL", "nunique"),
            second_level=("SECOND_LEVEL", "nunique")
        )
    )

    print(resumen)

    # inconsistencias jerárquicas
    if {"PRODUCTO_H", "FIRST_LEVEL", "SECOND_LEVEL"}.issubset(df.columns):

        inconsistencias = (
            df.groupby(["PRODUCTO_H", "FIRST_LEVEL", "SECOND_LEVEL"])
            .size()
            .reset_index(name="count")
        )

        problemas = inconsistencias.groupby("PRODUCTO_H").size()
        problemas = problemas[problemas > 1]

        if not problemas.empty:
            print("\n⚠️ Productos con múltiples jerarquías:")
            print(problemas.head())

    return resumen


# ==========================================
# 🔹 CARGA Y EXPORT
# ==========================================
def load_ocu_parquet(engine, year="2026", max_workers=4, chunksize=250_000, output_dir="exports"):

    os.makedirs(output_dir, exist_ok=True)

    print("\n================== CARGA OCU ==================")

    ocu_total = read_tables_parallel(
        engine=engine,
        year=year,
        max_workers=max_workers,
        chunksize=chunksize
    )

    if ocu_total.empty:
        print("⚠️ Sin datos")
        return ocu_total

    validate_data(ocu_total)

    path = os.path.join(output_dir, f"OCU_{year}_TOTAL.parquet")

    ocu_total.to_parquet(
        path,
        engine="pyarrow",
        index=False
    )

    print(f"\n💾 Exportado a: {path}")
    print(f"📊 Filas: {len(ocu_total):,}")

    return ocu_total


# ==========================================
# 🔹 EJECUCIÓN
# ==========================================
if __name__ == "__main__":

    ocu_total = load_ocu_parquet(
        engine=engine,   # 👈 tu conexión
        year="2026",
        max_workers=4,
        chunksize=250_000,
        output_dir="exports"
    )

In [ ]:
import pandas as pd
import os
import re
from sqlalchemy import inspect
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# 🔹 NORMALIZACIÓN DE COLUMNAS
# ==========================================
def normalize_columns(chunk):
    """
    - Convierte columnas a MAYÚSCULA
    - Resuelve duplicados tipo first_level vs FIRST_LEVEL
    - Prioriza columnas homologadas (minúsculas originales)
    """

    # Guardar columnas originales
    original_cols = chunk.columns.tolist()

    # Mapeo upper
    upper_cols = {col: col.upper() for col in chunk.columns}
    chunk.rename(columns=upper_cols, inplace=True)

    # =========================================
    # 🔥 REGLA DE NEGOCIO JERARQUÍAS
    # =========================================
    # Si existían ambas versiones, priorizamos la que venía en minúscula (homologada)

    if "FIRST_LEVEL" in chunk.columns:
        # buscar si existía versión minúscula original
        if "first_level" in original_cols:
            diff = (chunk["FIRST_LEVEL"] != chunk["FIRST_LEVEL"]).sum()
            # realmente aquí ya se sobreescribió, pero dejamos hook para logs

    if "SECOND_LEVEL" in chunk.columns:
        if "second_level" in original_cols:
            pass

    return chunk


# ==========================================
# 🔹 FUNCIÓN PARA LEER TABLA POR CHUNKS
# ==========================================
def read_table_in_chunks(table, engine, chunksize=500_000):

    dfs = []

    with engine.connect() as conn:
        query = f"SELECT * FROM [dbo].[{table}]"
        print(f"\n➡️ Leyendo tabla: {table} ...")

        # Detectar periodo desde nombre
        match = re.search(r"(20\d{2})[_\-]?(\d{2})", table)
        if match:
            year, month = match.groups()
            periodo_detectado = pd.to_datetime(f"{year}-{month}-01")
            print(f"   📅 Periodo detectado: {periodo_detectado}")
        else:
            periodo_detectado = pd.NaT
            print("   ⚠️ No se detectó periodo")

        for i, chunk in enumerate(pd.read_sql(query, conn, chunksize=chunksize)):
            print(f"   🧩 Chunk {i+1} ({len(chunk):,} filas)")

            # =========================================
            # 🔹 NORMALIZAR COLUMNAS
            # =========================================
            chunk = normalize_columns(chunk)

            # =========================================
            # 🔹 METADATOS
            # =========================================
            chunk["SOURCE_TABLE"] = table
            chunk["PERIODO"] = periodo_detectado

            dfs.append(chunk)

    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"✅ Tabla {table}: {len(df_final):,} filas")
        return df_final
    else:
        return pd.DataFrame()


# ==========================================
# 🔹 LECTURA PARALELA
# ==========================================
def read_tables_parallel(prefix, engine, max_workers=4, chunksize=500_000):

    inspector = inspect(engine)
    all_tables = inspector.get_table_names(schema="dbo")

    matching_tables = [
        t for t in all_tables
        if t.startswith(prefix) and not t.lower().endswith("bk")
    ]

    if not matching_tables:
        print(f"⚠️ No hay tablas con prefijo {prefix}")
        return pd.DataFrame()

    print(f"\n📦 Tablas encontradas ({len(matching_tables)}):")
    for t in matching_tables:
        print(f"   • {t}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(
            executor.map(
                lambda t: read_table_in_chunks(t, engine, chunksize),
                matching_tables
            )
        )

    df_total = pd.concat(dfs, ignore_index=True)
    print(f"\n🚀 Total cargado: {len(df_total):,} filas")

    return df_total


# ==========================================
# 🔹 VALIDACIÓN DE CALIDAD
# ==========================================
def validate_data(df):

    print("\n📊 VALIDACIÓN DE DATOS")

    resumen = (
        df.groupby("PERIODO")
        .agg(
            registros=("CLIENTE_ID","count"),
            first_level=("FIRST_LEVEL","nunique"),
            second_level=("SECOND_LEVEL","nunique")
        )
    )

    print(resumen)

    # detectar productos con múltiples jerarquías
    inconsistencias = (
        df.groupby(["PRODUCTO_H","FIRST_LEVEL","SECOND_LEVEL"])
        .size()
        .reset_index(name="count")
    )

    problemas = inconsistencias.groupby("PRODUCTO_H").size()
    problemas = problemas[problemas > 1]

    if not problemas.empty:
        print("\n⚠️ Productos con múltiples jerarquías:")
        print(problemas.head())

    return resumen


# ==========================================
# 🔹 CARGA Y EXPORT A PARQUET
# ==========================================
def load_ocu_parquet(engine, max_workers=4, chunksize=250_000, output_dir="exports"):

    os.makedirs(output_dir, exist_ok=True)

    print("\n================== CARGA OCU ==================")

    ocu_total = read_tables_parallel(
        prefix="OCU_2026",
        engine=engine,
        max_workers=max_workers,
        chunksize=chunksize
    )

    if ocu_total.empty:
        print("⚠️ Sin datos")
        return ocu_total

    # =========================================
    # 🔹 VALIDACIÓN FINAL
    # =========================================
    validate_data(ocu_total)

    # =========================================
    # 🔹 EXPORT
    # =========================================
    path = os.path.join(output_dir, "OCU_2026_TOTAL.parquet")

    ocu_total.to_parquet(
        path,
        engine="pyarrow",
        index=False
    )

    print(f"\n💾 Exportado a: {path}")
    print(f"📊 Filas: {len(ocu_total):,}")

    return ocu_total


# ==========================================
# 🔹 EJECUCIÓN
# ==========================================
if __name__ == "__main__":

    ocu_total = load_ocu_parquet(
        engine=engine,
        max_workers=4,
        chunksize=250_000,
        output_dir="exports"
    )

In [ ]:
import pandas as pd
import os
import re
from sqlalchemy import inspect
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# 🔹 CONFIGURACIÓN GLOBAL
# ==========================================
REQUIRED_COLUMNS = [
    "CLIENTE_ID",
    "PRODUCTO_H",
    "FIRST_LEVEL",
    "SECOND_LEVEL",
    "SEGMENTO_2021"
]


# ==========================================
# 🔹 NORMALIZACIÓN DE COLUMNAS
# ==========================================
def normalize_columns(chunk):
    """
    - Convierte columnas a MAYÚSCULA
    - Evita problemas de duplicados por casing
    """

    original_cols = chunk.columns.tolist()

    # Mapear a mayúscula
    chunk.columns = [col.upper() for col in chunk.columns]

    return chunk


# ==========================================
# 🔹 ASEGURAR COLUMNAS (ANTI-DRIFT)
# ==========================================
def ensure_columns(chunk):
    """
    Garantiza que todas las columnas críticas existan
    aunque alguna tabla no las traiga
    """

    for col in REQUIRED_COLUMNS:
        if col not in chunk.columns:
            chunk[col] = None

    return chunk


# ==========================================
# 🔹 VALIDACIÓN DE COLUMNAS
# ==========================================
def validate_columns(df):

    print("\n🧪 VALIDACIÓN DE COLUMNAS")

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]

    if missing:
        print("\n⚠️ Columnas faltantes:")
        for col in missing:
            print(f"   ❌ {col}")
    else:
        print("✅ Todas las columnas requeridas existen")

    print("\n📌 Total columnas:", len(df.columns))
    print("📌 Columnas disponibles:")
    print(sorted(df.columns))


# ==========================================
# 🔹 FUNCIÓN PARA LEER TABLA POR CHUNKS
# ==========================================
def read_table_in_chunks(table, engine, chunksize=500_000):

    dfs = []

    with engine.connect() as conn:
        query = f"SELECT * FROM [dbo].[{table}]"
        print(f"\n➡️ Leyendo tabla: {table} ...")

        # Detectar periodo
        match = re.search(r"(20\d{2})[_\-]?(\d{2})", table)
        if match:
            year, month = match.groups()
            periodo_detectado = pd.to_datetime(f"{year}-{month}-01")
            print(f"   📅 Periodo detectado: {periodo_detectado}")
        else:
            periodo_detectado = pd.NaT
            print("   ⚠️ No se detectó periodo")

        for i, chunk in enumerate(pd.read_sql(query, conn, chunksize=chunksize)):
            print(f"   🧩 Chunk {i+1} ({len(chunk):,} filas)")

            # 🔹 Normalizar columnas
            chunk = normalize_columns(chunk)

            # 🔹 Asegurar columnas críticas
            chunk = ensure_columns(chunk)

            # 🔹 Metadatos
            chunk["SOURCE_TABLE"] = table
            chunk["PERIODO"] = periodo_detectado

            dfs.append(chunk)

    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"✅ Tabla {table}: {len(df_final):,} filas")
        return df_final
    else:
        return pd.DataFrame()


# ==========================================
# 🔹 LECTURA PARALELA
# ==========================================
def read_tables_parallel(prefix, engine, max_workers=4, chunksize=500_000):

    inspector = inspect(engine)
    all_tables = inspector.get_table_names(schema="dbo")

    matching_tables = [
        t for t in all_tables
        if t.startswith(prefix) and not t.lower().endswith("bk")
    ]

    if not matching_tables:
        print(f"⚠️ No hay tablas con prefijo {prefix}")
        return pd.DataFrame()

    print(f"\n📦 Tablas encontradas ({len(matching_tables)}):")
    for t in matching_tables:
        print(f"   • {t}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(
            executor.map(
                lambda t: read_table_in_chunks(t, engine, chunksize),
                matching_tables
            )
        )

    # 🔥 CONCAT ROBUSTO (ANTI-DRIFT)
    df_total = pd.concat(dfs, ignore_index=True, sort=False)

    print(f"\n🚀 Total cargado: {len(df_total):,} filas")

    return df_total


# ==========================================
# 🔹 VALIDACIÓN DE CALIDAD
# ==========================================
def validate_data(df):

    print("\n📊 VALIDACIÓN DE DATOS")

    resumen = (
        df.groupby("PERIODO")
        .agg(
            registros=("CLIENTE_ID", "count"),
            first_level=("FIRST_LEVEL", "nunique"),
            second_level=("SECOND_LEVEL", "nunique")
        )
    )

    print(resumen)

    # detectar productos con múltiples jerarquías
    inconsistencias = (
        df.groupby(["PRODUCTO_H", "FIRST_LEVEL", "SECOND_LEVEL"])
        .size()
        .reset_index(name="count")
    )

    problemas = inconsistencias.groupby("PRODUCTO_H").size()
    problemas = problemas[problemas > 1]

    if not problemas.empty:
        print("\n⚠️ Productos con múltiples jerarquías:")
        print(problemas.head())

    return resumen


# ==========================================
# 🔹 CARGA Y EXPORT A PARQUET
# ==========================================
def load_ocu_parquet(engine, max_workers=4, chunksize=250_000, output_dir="exports"):

    os.makedirs(output_dir, exist_ok=True)

    print("\n================== CARGA OCU ==================")

    ocu_total = read_tables_parallel(
        prefix="OCU_2026",
        engine=engine,
        max_workers=max_workers,
        chunksize=chunksize
    )

    if ocu_total.empty:
        print("⚠️ Sin datos")
        return ocu_total

    # 🔹 VALIDACIONES
    validate_columns(ocu_total)
    validate_data(ocu_total)

    # 🔹 EXPORT
    path = os.path.join(output_dir, "OCU_2026_TOTAL.parquet")

    ocu_total.to_parquet(
        path,
        engine="pyarrow",
        index=False
    )

    print(f"\n💾 Exportado a: {path}")
    print(f"📊 Filas: {len(ocu_total):,}")

    return ocu_total


# ==========================================
# 🔹 EJECUCIÓN
# ==========================================
if __name__ == "__main__":

    ocu_total = load_ocu_parquet(
        engine=engine,
        max_workers=4,
        chunksize=250_000,
        output_dir="exports"
    )


# BASES GROSS Y CLIENTES UNICOS

In [4]:
import pandas as pd
from sqlalchemy import create_engine

# Suponiendo que ya tienes tu engine
# engine = create_engine("...tu conexión...")

query = """
SELECT *
FROM cierre_b2b_consolidado
WHERE ANO_INGRESO_RETIRO IN (2025, 2026);
"""

gross = pd.read_sql(query, engine)

# Revisar resultados
print(gross.head(10))
print(gross.columns)


  ANO_INGRESO_RETIRO MES_INGRESO_RETIRO FECHA_INGRESO_RETIRO        LINEA  \
0               2025              Enero           2025-01-01          Voz   
1               2025              Enero           2025-01-01     Internet   
2               2025              Enero           2025-01-01          Voz   
3               2025              Enero           2025-01-01          Voz   
4               2025              Enero           2025-01-01     Internet   
5               2025              Enero           2025-01-01     Internet   
6               2025              Enero           2025-01-01     Internet   
7               2025              Enero           2025-01-01          Voz   
8               2025              Enero           2025-01-01  Data Center   
9               2025              Enero           2025-01-01  Data Center   

        PRODUCTO PEDIDO_ID SUBPEDIDO_ID SOLICITUD_ID IDENTIFICADOR_ID  \
0      Telefonia      None         None         None       6043771151   
1    B

In [5]:
gross.to_csv("Gross_Churn.csv")

In [6]:
gross.to_parquet("gross_churn.parquet", index=False)

In [7]:
gross = pd.read_parquet("gross_churn.parquet")
print(gross.columns)

Index(['ANO_INGRESO_RETIRO', 'MES_INGRESO_RETIRO', 'FECHA_INGRESO_RETIRO',
       'LINEA', 'PRODUCTO', 'PEDIDO_ID', 'SUBPEDIDO_ID', 'SOLICITUD_ID',
       'IDENTIFICADOR_ID', 'CLIENTE_ID', 'NOMBRE_CLIENTE', 'GERENCIA',
       'REGIONAL', 'PLAZA', 'INTERFAZ', 'TECNOLOGIA_ID', 'nivel_3', 'nivel_2',
       'TIPO_TRANSACCION', 'TIPO_CANAL', 'CANTIDADF', 'VALOR', 'TIPO',
       'SUBTIPO', 'AGRUPADOR_ID', 'NOMBRE_EJECUTIVO', 'TIPO_CLIENTE',
       'GERENCIA_NUEVA', 'MARCA', 'CANAL_N1', 'CANAL_N2', 'CANAL_N3',
       'canal_id', 'Nombre_canal', 'ejecutivo_id', 'SUBPRODUCTO',
       'PLAN_FACTURA_ID', 'SEGMENTO_2021', 'FIRST_LEVEL', 'SECOND_LEVEL',
       'CLIENTE_ID_HOMOLOGADO', 'crosselling_familia', 'MOTIVO',
       'instituciones_educativas', 'EJECUTIVO_ID_VENTA',
       'NOMBRE_EJECUTIVO_VENTA', 'CANAL_ID_VENTA', 'NOMBRE_CANAL_VENTA',
       'LIDER_CATEGORIA_VENTA', 'LIDER_PLAZA_VENTA', 'LIDER_REGIONAL_VENTA',
       'TIPO_CANAL_VENTA', 'MUNICIPIO', 'DIRECCION', 'SUBMOTIVO_RETIRO',
      

In [8]:
gross["FECHA_INGRESO_RETIRO"].value_counts()

FECHA_INGRESO_RETIRO
2026-01-01 00:00:00    24902
2025-01-01 00:00:00    17534
2026-03-31 00:00:00    11110
2026-04-30 00:00:00    10851
2026-02-28 00:00:00    10670
                       ...  
2026-05-13 11:58:00        1
2026-05-03 21:09:06        1
2026-05-13 08:57:00        1
2026-05-22 08:35:51        1
2026-05-25 12:04:55        1
Name: count, Length: 73150, dtype: int64

In [9]:
import pandas as pd

query = """
SELECT *
FROM clientes_homologados
WHERE PERIODO >= '2025-01-01'
  AND PERIODO < '2027-01-01';
"""

df = pd.read_sql(query, engine)

print(df.head())
print(df.columns)


  SEGMENTO_2021  CLIENTE_ID CONVERGENCIA INTERFACES PERIODO NUEVO_SEGMENTO  \
0   Micro/Small  9016029432         FIJO       Open  202508       Medianas   
1   Micro/Small  9001337042   FIJO-MOVIL  Open-TIGO  202508       Medianas   
2   Micro/Small  1147950901         FIJO       Open  202508  Emprendedores   
3   Micro/Small   900223651         FIJO       Open  202508       Medianas   
4   Micro/Small  8110176948        MOVIL       TIGO  202508  Emprendedores   

  SEGMENTO_ANTERIOR  
0          Medianas  
1          Medianas  
2     Emprendedores  
3          Medianas  
4     Emprendedores  
Index(['SEGMENTO_2021', 'CLIENTE_ID', 'CONVERGENCIA', 'INTERFACES', 'PERIODO',
       'NUEVO_SEGMENTO', 'SEGMENTO_ANTERIOR'],
      dtype='object')


In [10]:
## cambiar periodo por date YYYY-MM-01
# Convertir PERIODO de YYYYMM a fecha YYYY-MM-01
df["PERIODO"] = pd.to_datetime(df["PERIODO"], format="%Y%m")

# Forzar el día 1
df["PERIODO"] = df["PERIODO"] + pd.offsets.Day(0)   # ya queda como YYYY-MM-01

print(df["PERIODO"].unique())
df.to_parquet("clientes_homologados.parquet", index=False)

<DatetimeArray>
['2025-08-01 00:00:00', '2025-09-01 00:00:00', '2025-10-01 00:00:00',
 '2025-03-01 00:00:00', '2025-04-01 00:00:00', '2025-07-01 00:00:00',
 '2025-01-01 00:00:00', '2026-02-01 00:00:00', '2026-03-01 00:00:00',
 '2026-04-01 00:00:00', '2025-06-01 00:00:00', '2025-11-01 00:00:00',
 '2025-02-01 00:00:00', '2025-05-01 00:00:00', '2026-01-01 00:00:00',
 '2025-12-01 00:00:00', '2026-05-01 00:00:00']
Length: 17, dtype: datetime64[ns]


In [11]:
df_leido = pd.read_parquet("clientes_homologados.parquet")

In [12]:
import pandas as pd

# Filtrar solo enero 2026
df_enero_2026 = df_leido[df_leido['PERIODO'] == pd.Timestamp('2026-05-01')]

# Contar clientes únicos por SEGMENTO_2021
clientes_unicos = df_enero_2026.groupby('SEGMENTO_2021')['CLIENTE_ID'].nunique()

# Mostrar resultado
print(clientes_unicos)

SEGMENTO_2021
Empresas                     5806
Gobierno                      499
Large                         976
Micro/Small                214121
Multinacionales               465
Wholesale International        48
Wholesale Others              305
Name: CLIENTE_ID, dtype: int64


# DEJAR UN SOLO ARCHIVO FACTURACION

In [1]:
import dask.dataframe as dd

# ==========================================
# 🔹 Rutas de entrada
# ==========================================
path_2025 = "exports/FACT_2025_TOTAL.parquet"
path_2026 = "exports/FACT_2026_TOTAL.parquet"

# ==========================================
# 🔹 Leer parquets con Dask
# ==========================================
print("📥 Leyendo FACT 2025...")
fact_2025 = dd.read_parquet(path_2025)

print("📥 Leyendo FACT 2026...")
fact_2026 = dd.read_parquet(path_2026)

# ==========================================
# 🔹 Número de filas (lazy)
# ==========================================
rows_2025 = fact_2025.shape[0].compute()
print(f"   Filas 2025: {rows_2025:,}")

rows_2026 = fact_2026.shape[0].compute()
print(f"   Filas 2026: {rows_2026:,}")

📥 Leyendo FACT 2025...
📥 Leyendo FACT 2026...
   Filas 2025: 17,567,685
   Filas 2026: 6,806,529


In [ ]:
df = fact_2025.compute()

In [2]:
df1 = fact_2026.compute()

In [6]:
#fact_2025 = df
fact_2026 =df1

In [ ]:
print(fact_2025.dtypes)
print(fact_2026.dtypes)

In [7]:
# ==========================
# Columnas problemáticas
# ==========================

NUMERIC_COLS = [
    "vlrCargoxFraccion",
    "vlrCargo"
]

DATE_COLS = [
    "PERIODO",
    "Fecha_Contable"
]


In [8]:
def normalize_columns(df):
    """
    Normaliza columnas críticas para permitir concat seguro
    """
    # Numéricas
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Fechas
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


In [9]:
import pandas as pd
# Normalizar cada año
#fact_2025 = normalize_columns(fact_2025)
fact_2026 = normalize_columns(fact_2026)

# Validación rápida
#print(fact_2025[NUMERIC_COLS + DATE_COLS].dtypes)
print(fact_2026[NUMERIC_COLS + DATE_COLS].dtypes)


C:\Users\ltequia\AppData\Local\Temp\ipykernel_19796\3352364876.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


vlrCargoxFraccion           float64
vlrCargo                    float64
PERIODO              datetime64[ns]
Fecha_Contable       datetime64[ns]
dtype: object


In [ ]:
chunk_size = 500_000  # ajusta según tu RAM

with open("FACT_TOTAL.csv", "w", newline="", encoding="utf-8") as f:
    for i in range(0, len(fact_2026), chunk_size):
        fact_2026.iloc[i:i+chunk_size].to_csv(
            f,
            index=False,
            header=(i == 0)
        )

In [ ]:
fact_total = pd.concat(
    [fact_2025, fact_2026],
    ignore_index=True
)
obj_cols = fact_total.select_dtypes(include="object").columns
fact_total[obj_cols] = fact_total[obj_cols].astype("string")

fact_total.to_parquet(
    "exports/FACT_TOTAL.parquet",
    engine="pyarrow",
    index=False
)



In [ ]:
# ==========================================
# 🔹 Concatenar
# ==========================================
fact_total = pd.concat(
    [fact_2025, fact_2026],
    ignore_index=True
)

print(f"\n📊 Total filas FACT consolidado: {len(fact_total):,}")

# ==========================================
# 🔹 Exportar parquet final
# ==========================================
output_path = "exports/FACT_TOTAL.parquet"

fact_total.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print(f"💾 FACT consolidado exportado a: {output_path}")


In [ ]:
print(fact_total.columns)

In [ ]:
# =========================================
# 1️⃣ Normalizar texto (por seguridad)
# =========================================

fact_total['SECOND_LEVEL'] = fact_total['SECOND_LEVEL'].astype(str).str.strip().str.upper()
fact_total['PRODUCTO_H'] = fact_total['PRODUCTO_H'].astype(str).str.strip().str.upper()

# =========================================
# 2️⃣ Tabla SECOND_LEVEL vs PRODUCTO_H
#     con suma de vlrCargo
# =========================================

tabla_second_producto = (
    fact_total
    .groupby(['second_LEVEL', 'PRODUCTO_H'], as_index=False)
    .agg(
        VLR_TOTAL=('vlrCargoxFraccion', 'sum'),
        REGISTROS=('vlrCargoxFraccion', 'count')
    )
    .sort_values(
        by=['SECOND_LEVEL', 'VLR_TOTAL'],
        ascending=[True, False]
    )
)

# =========================================
# 3️⃣ Mostrar tabla completa
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

tabla_second_producto

In [ ]:
# =========================================
# 1️⃣ Normalizar columnas
# =========================================

fact_total['SECOND_LEVEL'] = fact_total['SECOND_LEVEL'].astype(str).str.strip().str.upper()
fact_total['PRODUCTO_H'] = fact_total['PRODUCTO_H'].astype(str).str.strip().str.upper()

# Reemplazar nulos para que no se pierdan en el análisis
fact_total['SECOND_LEVEL'] = fact_total['SECOND_LEVEL'].fillna('SIN_SECOND_LEVEL')
fact_total['PRODUCTO_H'] = fact_total['PRODUCTO_H'].fillna('SIN_PRODUCTO')

# =========================================
# 2️⃣ Tabla agregada
# =========================================

tabla_second_producto = (
    fact_total
    .groupby(['SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        VLR_TOTAL=('vlrCargoxFraccion','sum'),
        REGISTROS=('vlrCargoxFraccion','count')
    )
)

# =========================================
# 3️⃣ Matriz SECOND_LEVEL vs PRODUCTO_H
# =========================================

matriz_second_producto = pd.pivot_table(
    tabla_second_producto,
    values='VLR_TOTAL',
    index='SECOND_LEVEL',
    columns='PRODUCTO_H',
    aggfunc='sum',
    fill_value=0
)

# =========================================
# 4️⃣ Ordenar por importancia económica
# =========================================

matriz_second_producto['TOTAL_SECOND_LEVEL'] = matriz_second_producto.sum(axis=1)
matriz_second_producto = matriz_second_producto.sort_values(
    by='TOTAL_SECOND_LEVEL',
    ascending=False
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

matriz_second_producto

In [ ]:
import os

print("Directorio actual:", os.getcwd())
print("Existe archivo:", os.path.exists("exports/OCU_TOTAL.parquet"))
print("Es archivo:", os.path.isfile("exports/OCU_TOTAL.parquet"))
print("Es carpeta:", os.path.isdir("exports/OCU_TOTAL.parquet"))

In [ ]:
import pandas as pd
import os

# ==========================================
# 🔹 Rutas de entrada
# ==========================================
import os

base_path = os.getcwd()
path_2025 = os.path.join(base_path, "exports", "OCU_2025_TOTAL.parquet")
path_2026 = os.path.join(base_path, "exports", "OCU_2026_TOTAL.parquet")

# ==========================================
# 🔹 Leer parquets
# ==========================================
OCU_2025 = pd.read_parquet(path_2025, engine="pyarrow")
print(f"   Filas: {len(OCU_2025):,}")

print("📥 Leyendo OCU 2026...")
OCU_2026 = pd.read_parquet(path_2026, engine="pyarrow")
print(f"   Filas: {len(OCU_2026):,}")

In [ ]:
#print(OCU_2025.dtypes)
print(OCU_2026.dtypes)

In [ ]:
print(OCU_2026.groupby("PERIODO")["CANTIDADF"].sum())

print(OCU_2025.groupby("PERIODO")["CANTIDADF"].sum())

In [ ]:
def normalize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )
    return df

In [ ]:
OCU_2025 = normalize_columns(OCU_2025)
OCU_2026 = normalize_columns(OCU_2026)

In [ ]:
print(OCU_2026.columns)
print(OCU_2025.columns)

In [ ]:
print(OCU_2025.columns[:5])
print(OCU_2026.columns[:5])

In [ ]:
cols_2025 = set(OCU_2025.columns)
cols_2026 = set(OCU_2026.columns)

print("Solo en 2025:", cols_2025 - cols_2026)
print("Solo en 2026:", cols_2026 - cols_2025)

In [ ]:
tabla_second_producto = (
    OCU_2026
    .groupby(['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        VLR_TOTAL=('CANTIDADF','sum'),
        CLIENTES_UNICOS=('CLIENTE_ID','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','VLR_TOTAL'],
        ascending=[True,True,True,False]
    )
)

# =========================================
# 3️⃣ Mostrar completo
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

tabla_second_producto

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import gc

# ==========================================================
# 1️⃣ CONFIGURACIÓN
# ==========================================================
output_file = "exports/OCU_TOTAL.parquet"
chunk_size = 1_000_000  # Ajusta según tu RAM

# Columnas numéricas
numeric_cols_forzar = [
    "CANTIDADF",
    "VALOR", "VALOR_1", "VALOR_2",
    "LATITUD", "LONGITUD",
    "PLAN_FACTURACION_ID",
    "MESES_ANTIGUEDAD",
    "ESTRATO"
]

print("🔧 Iniciando proceso...")

# ==========================================================
# 2️⃣ HOMOLOGAR COLUMNAS
# ==========================================================
common_cols = sorted(set(OCU_2025.columns) & set(OCU_2026.columns))

OCU_2025 = OCU_2025[common_cols]
OCU_2026 = OCU_2026[common_cols]

print(f"\n🔗 Columnas comunes: {len(common_cols)}")

# ==========================================================
# 3️⃣ FUNCIÓN DE TIPADO (RESPETANDO TU LÓGICA)
# ==========================================================
def aplicar_tipos(df, numeric_cols):

    numeric_cols = [c for c in numeric_cols if c in df.columns]

    # Numéricos
    for col in numeric_cols:
        print(f"   ➤ Numérico: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Fecha
    if "PERIODO" in df.columns:
        print("   ➤ Fecha: PERIODO")
        df["PERIODO"] = pd.to_datetime(df["PERIODO"], errors="coerce")

    # Strings (igual que tu script original)
    for col in df.columns:
        if col not in numeric_cols and col != "PERIODO":
            df[col] = df[col].astype("string")

    return df


# ==========================================================
# 4️⃣ APLICAR TIPOS
# ==========================================================
print("\n⚙️ Tipando OCU_2025...")
OCU_2025 = aplicar_tipos(OCU_2025, numeric_cols_forzar)

print("\n⚙️ Tipando OCU_2026...")
OCU_2026 = aplicar_tipos(OCU_2026, numeric_cols_forzar)

gc.collect()

# ==========================================================
# 5️⃣ ESCRITURA POR CHUNKS (CLAVE 🔥)
# ==========================================================
print("\n💾 Exportando a Parquet por chunks...")

writer = None

def escribir_chunks(df, writer):

    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i+chunk_size]

        # Convertir SOLO el chunk (evita el error de memoria)
        table = pa.Table.from_pandas(chunk, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)

        print(f"   ✔ Filas {i:,} - {i+len(chunk):,}")

    return writer


# 👉 IMPORTANTE: NO concatenamos
writer = escribir_chunks(OCU_2025, writer)

del OCU_2025
gc.collect()

writer = escribir_chunks(OCU_2026, writer)

del OCU_2026
gc.collect()

# Cerrar writer
if writer:
    writer.close()

print(f"\n✅ Archivo generado: {output_file}")


# ==========================================================
# 6️⃣ VALIDACIÓN LIGERA
# ==========================================================
import pyarrow.parquet as pq

pf = pq.ParquetFile(output_file)

print("\n📊 Validación:")
print(f"   ➤ Filas: {pf.metadata.num_rows:,}")
print(f"   ➤ Columnas: {pf.metadata.num_columns}")

In [ ]:
output_file = "exports/OCU_TOTAL.parquet"

In [ ]:
import pandas as pd

df = pd.read_parquet("exports/OCU_TOTAL.parquet")

df["PERIODO"] = df["PERIODO"].astype("category")

res = df.groupby("PERIODO", sort=False)["CANTIDADF"].sum()

print(res)